# Векторний пошук

В цьому ноутбуці ми розберемо вбудовування та векторний пошук. Ми розглянемо наступні питання:

- використання пакету sentence-transformers для вбудовування текстів;
- як робити векторний пошук вручну (для кращого розуміння теми);
- як використати minsearch для векторного пошуку;
- як інтегрувати векторний пошук (с minsearch) в RAG pipeline;
- як здійснювати векторний пушук за допомогою SQLite.

## Вибір моделі

Пакет sentence-transformers підтримує багато моделей. Вибір правильної моделі залежить від завдання, мови та наявних ресурсів. Більші моделі зазвичай повільніші, тому для набору даних поширених запитань коротких англійських текстів достатньо невеликої моделі. На власних даних треба спробувати кілька моделей і залишити ту, яка працює найкраще.

Для курсу ми використаємо модель all-MiniLM-L6-v2. Вона

- забезпечує 384-вимірні вектори (компактні)
- швидка на процесорі
- забезпечує гарну якість для загального англійського тексту
- використовує косинусну подібність

Модель all-MiniLM-L6-v2 видає нормалізовані вектори – вектори з одиничною довжиною. Коли обидва вектори нормалізовані, скалярний добуток дорівнює косинусній подібності. Ось чому в документації моделі зазначено, що вона «використовує косинусну подібність».

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Під час першого запуску програма завантажує модель (~80 МБ) та токенізатор з HuggingFace. Токенізатор перетворює текст на те, що модель може прочитати. Після цього обидва файли завантажуються з локального кешу.

## Вбудовування набору даних

Завантажимо в папку модуля 2 файл ingest.py, що створений у модулі 1 і який служить для завантаження FAQу.

In [3]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-08-05 12:37:45--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 738 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     738  --.-KB/s    in 0s      

2026-08-05 12:37:45 (44.6 MB/s) - ‘ingest.py’ saved [738/738]



Завантажимо FAQ.

In [2]:
from ingest import load_faq_data

documents = load_faq_data()

Подивимось на цей набір ще раз.

In [7]:
import json
print(json.dumps(documents[:2], ensure_ascii=False, indent=2))

[
  {
    "id": "9e508f2212",
    "course": "data-engineering-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Course: When does the course start?",
    "answer": "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."
  },
  {
    "id": "bfafa427b3",
    "course": "data-engineering-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Course: What are the prerequisites for this course?",
    "answer": "To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n-

Кожен документ - це словник Python, що містить питання та відповідь на нього. Ми вбудовуємо питання і відповідь разом. Таким чином, запит може зіставлятися з текстом питання та текстом відповіді в нашому індексі.

Створимо для кожного документу один текст.

In [3]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [4]:
print(f"Loaded {len(texts)} documents.")

Loaded 1401 documents.


Тепер ми генеруємо вбудовування. Ми маємо близько 1400 текстів. Ми не будемо передавати моделі усі одразу. Це займе багато часу, і ми не можемо бачити, що відбувається всередині. Натомість ми розділимо їх на групи (пакети).

Спочатку імпортуємо tqdm, щоб спостерігати за прогресом:

In [5]:
from tqdm.auto import tqdm

Далі ми розділяємо набір даних на пакети (батчі) по 50 текстів в кожному і кодуємо кожний пакет:

In [6]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/29 [00:00<?, ?it/s]

1401

Перетворюємо список векторів на двомірний масив (матрицю), де

- рядки - це документи (вектори);
- стовпці - це розміри векторів

In [7]:
import numpy as np
X = np.array(vectors)

Перевіримо розмірність.

In [8]:
X.shape

(1401, 384)

## Векторний пошук вручну

У нас є матриця X з усіма вбудованими документами. Ми беремо запит, порівнюємо його з кожним документом і залишаємо найсхожіші.

Коли надходить запит, ми його вбудовуємо:

In [17]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

Далі ми обчислюємо скалярний добуток для всіх документів:

In [18]:
scores = X.dot(v_query)

Це множення матриць на вектор. Результатом є список. Кожен елемент цього списку є scores - косинусною подібністю між документом i (рядком i) матриці X та v_query.

Найвищщий бал отримає найбільш схожий документ.

In [19]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

Це документ з номером 2. Подивимось на нього.

In [22]:
print(json.dumps(documents[idx], ensure_ascii=False, indent=2))

{
  "id": "3f1424af17",
  "course": "data-engineering-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "Course: Can I still join the course after the start date?",
  "answer": "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."
}


Зазвичай нам потрібно більше, ніж один найкращий збіг. Тому виберемо топ-5. 
np.argsort сортується від нижчого до вищого, тож останні 5 — це найвищі:

In [25]:
top5 = np.argsort(scores)[-5:]
top5

array([   7, 1009,  567, 1150,    2])

Найкращий буде в самому кінці списку. Отже варто перевернути його, щоб найкращий результат був першим.

In [27]:
top5 = top5[::-1]
top5

array([   2, 1150,  567, 1009,    7])

Тепер можна прочитати 5 найкращих результатів.

In [28]:
scores[top5]

array([0.762941  , 0.7579372 , 0.7192131 , 0.6536311 , 0.56009984],
      dtype=float32)

Всі ці операції можна записати одним рядком. Ось так:

In [29]:
top5 = np.argsort(-scores)[:5]

Це поширений спосіб перетворити сортування від меншого до більшого на сортування від більшого до меншого.

Прочитаємо знайдені нами документи.

In [30]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

Ми повертаємо 5 результатів, а не один найкращий тому, що в ньому може міститися не вся відповідь на питання. Вона може бути розподілена між кількома документами. Перший результат може виявитися неправильним. А правильний буде в іншому. Тому ми надсилаємо LLM всі 5, а вона вже їх об'єднує.

## Векторний пошук з Minsearch

В попередньому розділі ми реалізували векторний пошук вручну. Це не дуже зручно. Бібліотека Minsearch має спеціальний клас для векторного пошуку. Працює все так само.

Створимо індекс.

In [11]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

Ми передаємо масив numpy X з усіма вбудовуваннями та списком документів як корисне навантаження. Параметр keyword_fields працює так само, як і в text Index, тому ми зможемо фільтрувати за курсом.

Пошукаємо відповідь на питання.

In [32]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

Під капотом він робить те саме, що ми щойно робили вручну. Він обчислює скалярний добуток між кожним вектором (після фільтрації) та нашим вектором запиту.

Подивимось на головний результат:

In [33]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

Як і у випадку з текстовим пошуком, ми можемо фільтрувати за ключовими словами. Це важливо для взаємодії з користувачем. Студенту LLM у Zoom Camp байдуже на відповіді з курсу інженерії даних. Тому ми спочатку звужуємо вибір до його курсу, а потім оцінюємо лише його.

In [34]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

## RAG та векторний пошук

В модулі 1 ми побудували конвеєр RAG з трьох кроків. На кроці пошуку ми використовували пошук за ключовими словами. Замінимо його тепер на векторний. Скопіюємо для початку в папку модуля 2 файл rag_helper.py з модуля 1. Нам треба буде перевизначити в метод search. Для цього ми створимо дочірній клас RAGVector, який буде перевизначати цей метод.

In [8]:
from rag_helper import RAGBase

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )


Метод __init__додає один додатковий аргумент, embedder, для трансформатора речень. Всередині search ми використовуємо його для перетворення запиту на вектор. Потім ми запитуємо vindex з цим вектором замість необробленого тексту. Все інше успадковується від RAGBase.

Створимо клієнт Gemini

In [9]:
from dotenv import load_dotenv
load_dotenv()

from google import genai
gemini_client = genai.Client()

Ініціалізуємо новий клас.

In [12]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=gemini_client,
)

In [13]:
query = "I just found out about the program, can I still sign up?"
vector_assistant.rag(query)

/workspaces/LLM_Zoomcamp_2026/02-vector-search/rag_helper.py:77: UserWarning: Interactions usage is experimental and may change in future versions.
  interaction = self.llm_client.interactions.create(


'Yes, you can still join. If you want to receive a certificate, you just need to ensure you submit your project while submissions are still being accepted. Additionally, you do not need to worry about registration—you can start learning and submitting homework (while the form is open) right away.'

## Векторний пошук та SQLite

In [14]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

sqlitesearch підтримує три режими:

- lsh(за замовчуванням): до 100 тис. векторів, випадкові проекції гіперплощин
- ivf - 10K-500K векторів, кластеризація за K-середніми
- hnsw: 10 тис.-1 млн+ векторів, графік близькості (найвища повнота)

Для нашого невеликого набору даних lsh це добре. Усі режими використовують двофазний пошук: приблизний пошук кандидатів, а потім точне повторне ранжування за косинусною подібністю.

In [15]:
vs_index.fit(vectors, documents)

Індексуємо базу. Індекс зберігається у файлі faq_vectors2.db. Спробуємо щось знайти.

In [16]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': 'c207b8614e',
  'course': 'data-engineering-zoomcamp',
 

Зробимо фільтрацію за курсом.

In [17]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  '

І закриваємо з'єднання, якщо воно нам більше не потрібне.

In [18]:
vs_index.close()